# Symptom NER evaluation

Evaluates the saved transformer on the held-out split. Token classification and exact entity spans are reported separately.

## Data and provenance

In [1]:
from pathlib import Path
import sys
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src').is_dir() and (p/'data').is_dir())
sys.path.insert(0,str(ROOT));sys.path.insert(0,str(ROOT/'results'))
from utils.evaluation import setup,save,meta
ROOT=setup(ROOT);OUT=ROOT/'results'
import pandas as pd,json,collections
df=pd.read_csv(ROOT/'data/ner/splits/ner_test.csv'); print('rows',len(df),'columns',df.columns.tolist()); print(df.get('source',pd.Series(dtype=str)).value_counts(dropna=False)); display(df.head())


rows 116 columns ['text', 'mentions', 'status', 'tokens', 'labels', 'source']
source
annotated    116
Name: count, dtype: int64


,text,mentions,status,tokens,labels,source
0,"I don't have swelling of stomach, but I do hav...","[""swelling of stomach"", ""belly pain""]","[""PRESENT"", ""PRESENT""]","[""I"", ""don"", ""'"", ""t"", ""have"", ""swelling"", ""of...","[""O"", ""O"", ""O"", ""O"", ""O"", ""B-SYMPTOM"", ""I-SYMP...",annotated
1,I have swollen blood vessels but no prominent ...,"[""swollen blood vessels"", ""prominent veins on ...","[""PRESENT"", ""ABSENT""]","[""I"", ""have"", ""swollen"", ""blood"", ""vessels"", ""...","[""O"", ""O"", ""B-SYMPTOM"", ""I-SYMPTOM"", ""I-SYMPTO...",annotated
2,His dischromic patches have become more notice...,"[""dischromic patches""]","[""PRESENT""]","[""His"", ""dischromic"", ""patches"", ""have"", ""beco...","[""O"", ""B-SYMPTOM"", ""I-SYMPTOM"", ""O"", ""O"", ""O"",...",annotated
3,She says there is no distention of abdomen now.,"[""distention of abdomen""]","[""ABSENT""]","[""She"", ""says"", ""there"", ""is"", ""no"", ""distenti...","[""O"", ""O"", ""O"", ""O"", ""O"", ""B-SYMPTOM"", ""I-SYMP...",annotated
4,Mild visual disturbances show up near bright l...,"[""visual disturbances""]","[""PRESENT""]","[""Mild"", ""visual"", ""disturbances"", ""show"", ""up...","[""O"", ""B-SYMPTOM"", ""I-SYMPTOM"", ""O"", ""O"", ""O"",...",annotated


## Strict span evaluation

Model labels are aligned to the data token sequence through offsets. Exact span boundaries count as correct. Any annotation source distribution remains a dataset limitation, not evidence of clinical generalization.

In [2]:
import torch
from transformers import AutoTokenizer,AutoModelForTokenClassification
from sklearn.metrics import accuracy_score,classification_report
def js(x): return json.loads(x) if isinstance(x,str) else x
def charspans(tags,offsets):
 out=[]; start=end=None
 for t,(a,b) in zip(tags+['O'],offsets+[(0,0)]):
  if t=='B-SYMPTOM':
   if start is not None: out.append((start,end))
   start,end=a,b
  elif t=='I-SYMPTOM' and start is not None: end=b
  elif start is not None: out.append((start,end)); start=end=None
 return out
try:
 md=ROOT/'models/ner/symptom_ner_model'; tok=AutoTokenizer.from_pretrained(str(md),local_files_only=True); model=AutoModelForTokenClassification.from_pretrained(str(md),local_files_only=True); dev=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); model.to(dev).eval(); gold=[];pred=[]; G=set();P=set()
 for ri,r in df.iterrows():
  text=str(r.text); toks=js(r.tokens); labs=js(r.labels); spans=[]; cur=0
  for t in toks:
   p=text.find(t,cur); spans.append((p,p+len(t)) if p>=0 else (0,0)); cur=max(cur,p+len(t)) if p>=0 else cur
  en=tok(text,return_offsets_mapping=True,return_tensors='pt',truncation=True); offsets=en.pop('offset_mapping')[0].tolist(); en={k:v.to(dev) for k,v in en.items()}
  with torch.inference_mode(): ids=model(**en).logits[0].argmax(-1).cpu().tolist()
  pp=[model.config.id2label.get(i,'O') for i,(a,b) in zip(ids,offsets) if a!=b]; oo=[(a,b) for a,b in offsets if a!=b]; gg=[]
  for a,b in oo:
   ix=next((j for j,(x,y) in enumerate(spans) if x<b and y>a),None);gg.append(labs[ix] if ix is not None else 'O')
  G|={(ri,*s) for s in charspans(gg,oo)};P|={(ri,*s) for s in charspans(pp,oo)};gold+=gg;pred+=pp
 tp=len(G&P);fp=len(P-G);fn=len(G-P);pr=tp/(tp+fp) if tp+fp else 0;rc=tp/(tp+fn) if tp+fn else 0; f=2*pr*rc/(pr+rc) if pr+rc else 0; metrics={'sentences':len(df),'token_accuracy':accuracy_score(gold,pred),'strict_span_precision':pr,'strict_span_recall':rc,'strict_span_f1':f,'gold_spans':len(G),'pred_spans':len(P),'device':str(dev)}
 print(metrics);print(classification_report(gold,pred,zero_division=0));save(OUT/'metrics/ner.json',meta(ROOT,'symptom_ner',metrics=metrics,checkpoint=str(md)));
except Exception as e: print('UNAVAILABLE:',type(e).__name__,str(e))


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

{'sentences': 116, 'token_accuracy': 0.9349535382416011, 'strict_span_precision': 0.6903225806451613, 'strict_span_recall': 0.4885844748858447, 'strict_span_f1': 0.5721925133689839, 'gold_spans': 219, 'pred_spans': 155, 'device': 'cuda'}
              precision    recall  f1-score   support

   B-SYMPTOM       0.97      0.68      0.80       219
   I-SYMPTOM       0.72      0.98      0.83       212
           O       0.99      0.98      0.99       968

    accuracy                           0.93      1399
   macro avg       0.89      0.88      0.87      1399
weighted avg       0.95      0.93      0.94      1399



## Error analysis

Strict span errors can reflect boundary and tokenization mismatches. A rule baseline is not claimed because annotations are not assured to align with canonical dictionary spans. Token metrics are not entity metrics.